In [31]:
import torch
from torch import nn
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import Dataset
# Data handling libraries
import json
import numpy as np
import pandas as pd
from pandas import json_normalize
import torch
import typing as t
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler

# Natural Language Processing (NLP) libraries
from nltk.corpus import stopwords

# Scikit-learn modeling libraries
from sklearn.dummy import DummyClassifier # For baseline model
from sklearn.feature_extraction.text import TfidfVectorizer # To convert text to numbers
from sklearn.linear_model import LogisticRegression # The classifier model
from sklearn.metrics import accuracy_score, classification_report # For evaluation
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score # For splitting and validating
from sklearn.pipeline import Pipeline # To chain processing steps
from matplotlib import pyplot as plt

In [32]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

Using device: cuda


In [33]:
encoder_text = "cmarkea/distilcamembert-base" # CamemBERT tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(encoder_text)
max_len = 128  # Maximum length for tokenization

In [34]:
# Load the training data from a JSON Lines file (one JSON object per line)
train_data = pd.read_json('../Data/train.jsonl', lines=True)

# The tweet data is nested. json_normalize flattens the nested JSON into columns.
train_data = json_normalize(train_data.to_dict(orient='records'))

# Load the Kaggle test data (which we will make predictions on)
kaggle_data = pd.read_json('../Data/kaggle_test.jsonl', lines=True)
# Also normalize the Kaggle data
kaggle_data = json_normalize(kaggle_data.to_dict(orient='records'))


# Separate features from the target variable for the training set
X_train_raw = train_data.drop('label', axis=1)
y_train_raw = train_data['label']

X_kaggle_raw = kaggle_data

In [37]:
additional_features = ['quoted_status.is_quote_status', 'quoted_status.quote_count', 'quoted_status.favorite_count',
                       'quoted_status.reply_count', 'quoted_status.user.friends_count', 'quoted_status.user.verified', 'quoted_status.user.statuses_count',
                       'quoted_status.user.followers_count', 'user.listed_count', 'user.favourites_count', 'user.geo_enabled', 'user.statuses_count', 
                       'user.default_profile', 'challenge_id']

In [41]:
def extract_full_text(tweet):
    # Start with the standard 'text' field
    text = tweet['text']
    # Check if the 'extended_tweet.full_text' field exists (is not NaN)
    if not pd.isna(tweet['extended_tweet.full_text']):
        # If it exists, it's the full text, so use it instead
        text = tweet['extended_tweet.full_text']
    return text

def simplify_source(src):
    if src is None:
        return "unknown"
    if str(src) == 'nan':
        return "unknown"
    s = src.lower()

    if "twitter" in s:
        return "twitter_official"
    if any(x in s for x in ["buffer", "hootsuite", "publer", "sprinklr", "tweetdeck", "ifttt", "post", "planable"]):
        return "scheduler"
    if any(x in s for x in ["bot", "auto", "rss", "feed", "revive", "publisher"]):
        return "automation"
    if any(x in s for x in ["instagram", "insta"]):
        return "instagram"
    if any(x in s for x in ["linkedin"]):
        return "linkedin"
    if any(x in s for x in ["press", "journal", "news", "mag", "africa", "france"]):
        return "media"
    
    return "other"

def extract_features(df: pd.DataFrame) -> pd.DataFrame:
    features = pd.DataFrame()

    # Source du tweet
    features["source_clean"] = df["source"].str.extract(r'>(.*?)<')
    features["source_group"] = features["source_clean"].apply(simplify_source)
    source_dummies = pd.get_dummies(features, columns=["source_group"]).drop(columns=["source_clean"])

    features["is_quote"] = df["quoted_status"].notnull().astype(int)

    # Date features
    features["hour"] = df["created_at"].dt.hour
    features["dayofweek"] = df["created_at"].dt.dayofweek

    features["full_text"] = df.apply(lambda tweet: extract_full_text(tweet), axis=1)

    features['is_url'] = df['user.url'].notna().astype(int)
    
    for feat in additional_features:
        features[feat] = df[feat].fillna(0).astype(int)
    
    return pd.concat([features.drop(columns=["source_clean", "source_group"]), source_dummies], axis=1)

In [42]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(extract_features(X_train_raw), y_train_raw, test_size=0.2, random_state=42)
X_train

,is_quote,hour,dayofweek,full_text,is_url,quoted_status.is_quote_status,quoted_status.quote_count,quoted_status.favorite_count,quoted_status.reply_count,quoted_status.user.friends_count,...,user.default_profile,challenge_id,source_group_automation,source_group_instagram,source_group_linkedin,source_group_media,source_group_other,source_group_scheduler,source_group_twitter_official,source_group_unknown
118805,0,11,5,@yvanmamalof @BioHospitalix Maintenant si 😅\n\...,0,0,0,0,0,0,...,1,197296,False,False,False,False,False,False,True,False
1199,0,14,5,@klorydryk La Liberté se défend mieux avec des...,0,0,0,0,0,0,...,1,2028,False,False,False,False,False,False,True,False
139926,0,17,0,Ai-je raison ??? #JeudiConfinement,0,0,149,906,107,1,...,1,232971,False,False,False,False,False,False,True,False
6650,0,13,2,La chance 😭,0,0,197,3331,88,44,...,0,11102,False,False,False,False,False,False,True,False
61708,0,11,1,Elle est en train oui,0,0,0,2,0,868,...,1,102568,False,False,False,False,False,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119879,0,11,3,Mdrr mais y'aura combien de vagues au juste ??...,0,0,63,263,59,617,...,1,199125,False,False,False,False,False,False,True,False
103694,0,13,5,PTTTDDDRRRRRR LA JE SUIS MORT,0,0,203,132,28,956,...,1,171926,False,False,False,False,False,False,True,False
131932,0,20,0,@ChrisMcCandl @rocky_kodio @cavousf5 Si tout v...,0,0,0,0,0,0,...,1,219537,False,False,False,False,False,False,True,False
146867,0,20,1,Élèves à moins de 2m@sans masque ... et le pro...,0,0,48,484,141,2517,...,1,244738,False,False,False,False,False,False,True,False


In [45]:
class TweetDataset(Dataset):
    def __init__(self, texts, meta, labels, tokenizer, max_len):
        self.texts = texts
        self.meta = meta
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        meta = self.meta[idx]

        enc = self.tokenizer(
            text,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "meta": torch.tensor(meta, dtype=torch.float),
            "label": torch.tensor(label, dtype=torch.float)
        }

In [46]:
scaler = StandardScaler()
X_meta_train_scaled = scaler.fit_transform(X_train.drop(columns=["full_text"]))
X_meta_test_scaled = scaler.transform(X_test.drop(columns=["full_text"]))

In [47]:
train_dataset = TweetDataset(
    texts=X_train["full_text"].tolist(),
    meta=X_meta_train_scaled,
    labels=y_train.tolist(),
    tokenizer=tokenizer,
    max_len=max_len
)

test_dataset = TweetDataset(
    texts=X_test["full_text"].tolist(),
    meta=X_meta_test_scaled,
    labels=y_test.tolist(),
    tokenizer=tokenizer,
    max_len=max_len
)

kaggle_dataset = TweetDataset(
    texts=X_kaggle_raw.apply(lambda tweet: extract_full_text(tweet), axis=1).tolist(),
    meta=scaler.transform(extract_features(X_kaggle_raw).drop(columns=["full_text"])),
    labels=[0]*len(X_kaggle_raw),  # Dummy labels for Kaggle data
    tokenizer=tokenizer,
    max_len=max_len
)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)
kaggle_loader = DataLoader(kaggle_dataset, batch_size=16, shuffle=False)

In [48]:
class CamembertClassifier(nn.Module):
    def __init__(self, meta_dim, num_classes=1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(encoder_text)
        hidden = self.encoder.config.hidden_size     # 768
        self.dropout = nn.Dropout(0.2)
        self.fc = nn.Linear(hidden + meta_dim, num_classes)

    def forward(self, input_ids, attention_mask, meta):
        # input shape : [batch_size, seq_len]
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        ) # shape [batch_size, seq_len, hidden_size]
        cls = outputs.last_hidden_state[:, 0, :]  # vecteur [CLS]
        # cls shape : [batch_size, hidden_size]
        x = self.dropout(cls)
        x = torch.cat((x, meta), dim=1)
        logits = self.fc(x)
        # logits shape : [batch_size, num_classes]
        return logits

In [49]:
model = CamembertClassifier(meta_dim=len(meta_cols)).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

Some weights of CamembertModel were not initialized from the model checkpoint at cmarkea/distilcamembert-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [50]:
def train_model(model, train_loader, test_loader, optimizer, loss_criterion, num_epochs):
    iter = 0
    history_train_acc, history_val_acc, history_train_loss, history_val_loss = [], [], [], []
    best_accuracy = 0
    for epoch in range(num_epochs):
        for i, batch in enumerate(train_loader):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device).unsqueeze(1)
            meta = batch['meta'].to(device)
            # Training mode
            model.train()

            # Clear gradients with respect to parameters
            optimizer.zero_grad()
            logits = model(input_ids, attention_mask, meta)

            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            iter += 1

            if iter % 100 == 0:
                train_loss = loss.data.item()

                model.eval()

                correct = 0
                total = 0
                with torch.no_grad():
                    for batch in test_loader:
                        input_ids = batch['input_ids'].to(device)
                        attention_mask = batch['attention_mask'].to(device)
                        labels = batch['label'].to(device).unsqueeze(1)
                        meta = batch['meta'].to(device)


                        outputs = model(input_ids, attention_mask, meta)  # (batch_size, nclasses)
                        
                        val_loss = loss_criterion(outputs, labels)

                        # multiclass prediction: choose argmax
                        probs = torch.sigmoid(outputs)
                        predicted = (probs > 0.5).float()
                        
                        total += labels.size(0)
                        correct += (predicted.cpu() == labels.cpu()).sum().item()
                
                accuracy = 100. * correct / total

                print(f'Iter: {iter:4} | Train Loss: {train_loss:.3f} | Val Loss: {val_loss.item():2.3f} | Val Accuracy: {accuracy:.2f}')
                history_val_loss.append(val_loss.data.item())
                history_val_acc.append(round(accuracy, 2))
                history_train_loss.append(train_loss)

                # Save model when accuracy beats best accuracy
                if accuracy > best_accuracy:
                    best_accuracy = accuracy
                    # We can load this best model on the validation set later
                    torch.save(model.state_dict(), 'best_model.pth')
    return (history_train_acc, history_val_acc, history_train_loss, history_val_loss)

In [51]:
train_model(
    model,
    train_loader,
    test_loader,
    optimizer,
    criterion,
    num_epochs=3
)

Iter:  100 | Train Loss: 0.617 | Val Loss: 0.961 | Val Accuracy: 61.81
Iter:  200 | Train Loss: 0.564 | Val Loss: 0.813 | Val Accuracy: 64.28
Iter:  300 | Train Loss: 0.738 | Val Loss: 0.842 | Val Accuracy: 63.31
Iter:  400 | Train Loss: 0.662 | Val Loss: 0.809 | Val Accuracy: 65.26
Iter:  500 | Train Loss: 0.702 | Val Loss: 0.723 | Val Accuracy: 67.09
Iter:  600 | Train Loss: 0.601 | Val Loss: 0.830 | Val Accuracy: 67.01
Iter:  700 | Train Loss: 0.509 | Val Loss: 0.826 | Val Accuracy: 67.41
Iter:  800 | Train Loss: 0.614 | Val Loss: 0.728 | Val Accuracy: 67.37
Iter:  900 | Train Loss: 0.507 | Val Loss: 0.780 | Val Accuracy: 67.43
Iter: 1000 | Train Loss: 0.532 | Val Loss: 0.751 | Val Accuracy: 67.19
Iter: 1100 | Train Loss: 0.451 | Val Loss: 0.701 | Val Accuracy: 67.61


KeyboardInterrupt: 

# Berth pas fine tuned + meta data into XGBoost 

In [52]:
class CamembertFeaturizer(nn.Module):
    def __init__(self, encoder_name, meta_dim):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(encoder_name)
        self.hidden = self.encoder.config.hidden_size  # 768 pour CamemBERT
        self.meta_dim = meta_dim

        # On fige BERT
        for p in self.encoder.parameters():
            p.requires_grad = False

    def forward(self, input_ids, attention_mask, meta):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        cls = outputs.last_hidden_state[:, 0, :]   # [batch, hidden]
        # Concatenation BERT + méta -> feature final pour XGBoost
        feats = torch.cat([cls, meta], dim=1)      # [batch, hidden + meta_dim]
        return feats

In [53]:
meta_dim = X_meta_train_scaled.shape[1]
encoder_text = "camembert-base"  # ou ta variable globale

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
featurizer = CamembertFeaturizer(encoder_text, meta_dim).to(device)
featurizer.eval()   # très important


CamembertFeaturizer(
  (encoder): CamembertModel(
    (embeddings): CamembertEmbeddings(
      (word_embeddings): Embedding(32005, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): CamembertEncoder(
      (layer): ModuleList(
        (0-11): 12 x CamembertLayer(
          (attention): CamembertAttention(
            (self): CamembertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): CamembertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
        

In [56]:
def extract_features(featurizer, dataloader, device):
    featurizer.eval()
    all_feats = []
    all_labels = []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            meta = batch["meta"].to(device)
            labels = batch["label"].cpu().numpy()

            feats = featurizer(input_ids, attention_mask, meta)  # [batch, hidden+meta_dim]
            all_feats.append(feats.cpu().numpy())
            all_labels.append(labels)

    X = np.vstack(all_feats)              # (N, hidden+meta_dim)
    y = np.concatenate(all_labels, axis=0)  # (N,)
    return X, y

X_train_feats, y_train_feats = extract_features(featurizer, train_loader, device)
X_val_feats,   y_val_feats   = extract_features(featurizer, test_loader,   device)

In [58]:
import xgboost as xgb

clf = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",    # "gpu_hist" si tu as un GPU compatible
    eval_metric="logloss"
)

clf.fit(X_train_feats, y_train_feats)

from sklearn.metrics import accuracy_score

y_val_pred = clf.predict(X_val_feats)
print("Val accuracy:", accuracy_score(y_val_feats, y_val_pred))


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.6/223.6 MB 56.6 MB/s  0:00:03m0:00:0100:01
Val accuracy: 0.8385566278281639


In [59]:
X_kaggle_feats, _ = extract_features(featurizer, kaggle_loader, device)
kaggle_preds = clf.predict(X_kaggle_feats)

In [61]:
output = pd.DataFrame({
        "ID": X_kaggle_raw['challenge_id'],
        "Prediction": kaggle_preds
    })

output.to_csv("submission.csv", index=False)
print("Submission saved.")

Submission saved.
